# 🚢 Titanic Passenger Survival Prediction (Binary Classification)

This project builds an end-to-end Machine Learning classification pipeline to predict passenger survival on the Titanic.
We cover:
- Exploratory Data Analysis (EDA) & Demographic Insights
- Missing Value Imputation (Age, Embarked, Cabin)
- Feature Engineering (Title Extraction, Family Size, IsAlone indicator)
- Categorical Encoding & Feature Scaling
- Model Training & Evaluation across multiple baseline classifiers (Logistic Regression, KNN, Decision Tree, Random Forest)
- Performance Diagnostics (Confusion Matrix, ROC-AUC, Feature Importances)


### 1. Library Imports
Importing necessary libraries:
- `pandas` & `numpy` for data manipulation and numerical arrays
- `matplotlib.pyplot` & `seaborn` for exploratory visualization
- `scikit-learn` for preprocessing, model training, and performance metrics


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')


### 2. Loading the Dataset
Load `Titanic-Dataset.csv` into a pandas DataFrame and inspect the first 5 records to understand the tabular schema.


In [ ]:
data = pd.read_csv("Titanic-Dataset.csv")
df = pd.DataFrame(data)
df.head()


### 3. Dataset Shape & Data Types
Examine the dimensions (rows, columns) and feature datatypes to identify numerical vs. categorical attributes.


In [ ]:
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\n--- Column Data Types ---")
print(df.dtypes)


### 4. Summary Statistics
Compute descriptive statistics (mean, std, min, quartiles, max) for numerical attributes (`Age`, `Fare`, `SibSp`, `Parch`).


In [ ]:
df.describe()


### 5. Missing Value Analysis
Identify columns containing null/missing values and calculate their percentage relative to the total dataset.


In [ ]:
missing_counts = df.isna().sum()
missing_percent = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_counts, 'Percentage (%)': missing_percent})
missing_df[missing_df['Missing Count'] > 0]


### 6. Target Variable Distribution (`Survived`)
Inspect the balance of the target variable (0 = Deceased, 1 = Survived) using value counts and a frequency countplot.


In [ ]:
survived_counts = df['Survived'].value_counts()
print("Survival Counts:\n", survived_counts)
print(f"Survival Rate: {survived_counts[1] / len(df) * 100:.2f}%")

plt.figure(figsize=(6, 4))
sns.countplot(x='Survived', data=df, palette='pastel')
plt.title('Distribution of Passenger Survival (0 = No, 1 = Yes)')
plt.xlabel('Survival Status')
plt.ylabel('Count')
plt.xticks([0, 1], ['Did Not Survive (0)', 'Survived (1)'])
plt.show()


### 7. Exploratory Analysis: Survival by Gender
Analyze how passenger gender impacted survival chances, reflecting the maritime protocol of prioritizing women and children.


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x='Sex', y='Survived', data=df, palette='Set2')
plt.title('Survival Rate by Gender')
plt.ylabel('Survival Probability')
plt.show()

pd.crosstab(df['Sex'], df['Survived'], normalize='index') * 100


### 8. Exploratory Analysis: Survival by Ticket Class (`Pclass`)
Examine socio-economic status vs. survival probability across 1st, 2nd, and 3rd passenger classes.


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x='Pclass', y='Survived', data=df, palette='Blues_d')
plt.title('Survival Rate by Passenger Class (Pclass)')
plt.xlabel('Passenger Class (1 = 1st, 2 = 2nd, 3 = 3rd)')
plt.ylabel('Survival Probability')
plt.show()


### 9. Exploratory Analysis: Age vs. Survival Distribution
Visualize passenger age distributions comparing survivors and non-survivors using kernel density estimates (KDE).


In [ ]:
plt.figure(figsize=(9, 4))
sns.kdeplot(df[df['Survived'] == 1]['Age'].dropna(), label='Survived', shade=True, color='teal')
sns.kdeplot(df[df['Survived'] == 0]['Age'].dropna(), label='Did Not Survive', shade=True, color='crimson')
plt.title('Age Distribution by Survival Status')
plt.xlabel('Age')
plt.ylabel('Density')
plt.legend()
plt.show()


### 10. Data Cleaning & Missing Value Imputation
- `Age`: Impute missing entries with the median age within respective passenger classes.
- `Embarked`: Impute the 2 missing ports with the mode ('S').
- `Cabin`: Over 77% missing; drop this column to avoid introducing noisy sparsity.
- Verify that no null values remain in the working dataset.


In [ ]:
# Impute Age with median grouped by Pclass
df['Age'] = df.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

# Impute Embarked with mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop Cabin due to high missing rate
df.drop(columns=['Cabin'], inplace=True)

print("Remaining Missing Values:")
print(df.isna().sum())


### 11. Feature Engineering: Title Extraction & Family Metrics
Extract informative representations from raw text:
1. `Title`: Parse passenger titles from `Name` (e.g., Mr, Mrs, Miss, Master) and group rare titles into 'Rare'.
2. `FamilySize`: Calculate total family members aboard as `SibSp + Parch + 1`.
3. `IsAlone`: Binary flag indicating whether the passenger traveled without family (`FamilySize == 1`).
4. Drop uninformative identifiers: `PassengerId`, `Name`, and `Ticket`.


In [ ]:
# Extract Title from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
                                   'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# Family Metrics
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Drop redundant identifier columns
df.drop(columns=['PassengerId', 'Name', 'Ticket'], inplace=True)

df.head()


### 12. Categorical Encoding
Convert categorical text features (`Sex`, `Embarked`, `Title`) into numeric binary dummy variables using `pd.get_dummies(..., drop_first=True)` to avoid the dummy variable trap.


In [ ]:
df_encoded = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)
print(f"Encoded DataFrame Shape: {df_encoded.shape}")
df_encoded.head()


### 13. Correlation Analysis
Generate a correlation heatmap across all numerical and encoded features to assess feature relationships with `Survived`.


In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df_encoded.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap with Engineered Features')
plt.show()


### 14. Train-Test Split with Stratification
Separate features ($X$) and target ($y = Survived$). Perform an 80/20 train-test split with `stratify=y` to preserve class ratios in both partitions.


In [ ]:
X = df_encoded.drop(columns=['Survived'])
y = df_encoded['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Samples: {X_train.shape[0]}, Test Samples: {X_test.shape[0]}")


### 15. Feature Normalization (`StandardScaler`)
Fit a `StandardScaler` strictly on the training partition and transform both `X_train` and `X_test` to prevent data leakage and place all numeric variables on standard normal scales ($z = (x - \mu)/\sigma$).


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### 16. Multi-Model Training & Benchmark
Train four diverse classification algorithms on the scaled training dataset:
1. **Logistic Regression** (Linear baseline probabilistic classifier)
2. **K-Nearest Neighbors** (Instance-based metric learner)
3. **Decision Tree Classifier** (Non-linear rule-based tree model)
4. **Random Forest Classifier** (Bagging ensemble meta-estimator)


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': roc
    })

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
results_df


### 17. Visual Performance Comparison
Plot comparative bar charts of Accuracy and F1-Score across the trained algorithms to identify the top performer.


In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(x='Model', y='Accuracy', data=results_df, palette='viridis')
plt.title('Classifier Accuracy Comparison on Titanic Test Set')
plt.ylabel('Accuracy Score')
plt.ylim(0.7, 0.9)
for index, row in results_df.iterrows():
    plt.text(index, row['Accuracy'] + 0.005, f"{row['Accuracy']:.3f}", ha='center', fontweight='bold')
plt.show()


### 18. Diagnostic Confusion Matrix: Random Forest
Compute and display the confusion matrix heatmap for the Random Forest classifier to analyze True Positives, True Negatives, False Positives, and False Negatives.


In [ ]:
rf_model = models['Random Forest']
y_rf_pred = rf_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_rf_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Deceased', 'Predicted Survived'],
            yticklabels=['Actual Deceased', 'Actual Survived'])
plt.title('Random Forest Confusion Matrix')
plt.ylabel('Ground Truth')
plt.xlabel('Prediction')
plt.show()

print("Classification Report:\n")
print(classification_report(y_test, y_rf_pred, target_names=['Did Not Survive', 'Survived']))


### 19. Random Forest Feature Importance
Analyze MDI (Mean Decrease in Impurity) feature importances from the Random Forest model to see which attributes contributed most heavily to survival predictions.


In [ ]:
feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
feature_importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importances for Titanic Survival (Random Forest)')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.show()


### 20. Conclusion & Key Takeaways
1. **Demographic Influence**: Gender (`Sex_male`), Ticket Class (`Pclass`), and extracted passenger titles (`Title_Mr`) were the most decisive indicators of survival.
2. **Missing Data Imputation**: Conditional group median imputation on `Age` based on passenger class prevented distortion while retaining records.
3. **Model Performance**: Ensemble methods (Random Forest) achieved strong generalization with balanced precision and recall on unseen validation data.
